# Option B: Semantic Segmentation — SAM2 한계 분석 + SegFormer 직접 구현

## 왜 SAM2가 mIoU=0.107에 머물렀나?

| | SAM2 (Phase 4에서 사용) | SegFormer (이번 구현) |
|---|---|---|
| **태스크** | Promptable Instance Segmentation | Semantic Segmentation |
| **출력** | 물체 경계 마스크 (class 없음) | 픽셀당 class ID |
| **CARLA GT** | 픽셀당 semantic class ID | 픽셀당 semantic class ID |
| **미스매치** | SAM2 마스크 ≠ semantic class 영역 | 동일한 형식 |

**결론**: SAM2는 '무엇이 있는지'를 모른다. 경계만 찾는다. CARLA는 '무엇인지'를 요구한다.
→ **올바른 도구**: SegFormer = Transformer 기반 Semantic Segmentation

## 구현 계획
1. CARLA semantic GT 분석 (클래스 분포, 픽셀 비율)
2. SegFormer-b0 (pretrained on ADE20K) → CARLA 클래스로 헤드 교체
3. CARLA 데이터로 파인튜닝
4. mIoU 평가 (Phase 4 SAM2 0.107과 비교)

## 핵심 논문
- **SegFormer** (Xie et al., NeurIPS 2021): Mix Transformer(MiT) 백본 + 경량 MLP 디코더
- 특징: 계층적 Transformer, 위치 인코딩 없음 → 임의 해상도 입력 가능

In [ ]:
# 한글 폰트 설정 (필수)
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import os, sys, random, json, warnings
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__}')

## Cell 1: CARLA Semantic GT 분석

In [ ]:
# ── 경로 설정 ──────────────────────────────────────────────────────────────
BASE_DIR  = Path(r'C:/Users/apple/Desktop/autonomous_cv_pipeline/phase4_carla')
DATA_DIR  = BASE_DIR / 'data_collection' / 'carla_dataset'
IMG_DIR   = DATA_DIR / 'images'
SEM_DIR   = DATA_DIR / 'semantic'
OUT_DIR   = BASE_DIR / 'finetuning'

# CARLA 0.9.15 semantic class 정의
CARLA_CLASSES = {
    0:  ('Unlabeled',   '#000000'),
    1:  ('Building',    '#70246e'),
    2:  ('Fence',       '#100035'),
    3:  ('Other',       '#90c0b0'),
    4:  ('Pedestrian',  '#220088'),
    5:  ('Pole',        '#111188'),
    6:  ('RoadLine',    '#ffffff'),
    7:  ('Road',        '#804080'),
    8:  ('SideWalk',    '#f423e8'),
    9:  ('Vegetation',  '#6b8e23'),
    10: ('Vehicles',    '#0000e8'),
    11: ('Wall',        '#64496e'),
    12: ('TrafficSign', '#331549'),
    13: ('Sky',         '#4682b4'),
    14: ('Ground',      '#81007f'),
    15: ('Bridge',      '#b05a21'),
    16: ('RailTrack',   '#0e3a1f'),
    17: ('GuardRail',   '#0f4942'),
    18: ('TrafficLight','#f0d058'),
    19: ('Static',      '#606060'),
    20: ('Dynamic',     '#6f0000'),
    21: ('Water',       '#55aa96'),
    22: ('Terrain',     '#98fb98'),
    28: ('Car',         '#0000ff'),  # CARLA 0.9.15 확장
}

# ── 클래스 분포 분석 ──────────────────────────────────────────────────────
sem_files = sorted(SEM_DIR.glob('*.png'))
print(f'Semantic 파일 수: {len(sem_files)}')

cls_pixel_count = Counter()
cls_frame_count = Counter()
SAMPLE_N = 100  # 전체 분석 시간 절약

for f in sem_files[:SAMPLE_N]:
    img = cv2.imread(str(f))
    r_ch = img[:, :, 2]  # R채널 = class ID
    unique, counts = np.unique(r_ch, return_counts=True)
    for cls_id, cnt in zip(unique, counts):
        cls_pixel_count[int(cls_id)] += int(cnt)
        cls_frame_count[int(cls_id)] += 1

total_pixels = sum(cls_pixel_count.values())
print(f'\n[{SAMPLE_N}프레임 분석 결과]')
print(f'{"클래스":>4} {"이름":>12} {"픽셀비율":>10} {"등장프레임":>10}')
print('-' * 45)
for cls_id, pix in sorted(cls_pixel_count.items(), key=lambda x: -x[1]):
    name = CARLA_CLASSES.get(cls_id, ('Unknown',))[0]
    pct  = pix / total_pixels * 100
    frm  = cls_frame_count[cls_id]
    print(f'{cls_id:>4} {name:>12} {pct:>9.2f}% {frm:>10}프레임')

# ── 샘플 시각화 ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('CARLA Semantic Segmentation GT 샘플', fontsize=13)

sample_files = random.sample(sem_files, 6)
for ax, f in zip(axes.flat, sample_files):
    img_path = IMG_DIR / f'{f.stem}.jpg'
    rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    sem = cv2.imread(str(f))
    r_ch = sem[:, :, 2]
    
    # 클래스별 컬러맵 적용
    colored = np.zeros((*r_ch.shape, 3), dtype=np.uint8)
    for cls_id, (name, hex_color) in CARLA_CLASSES.items():
        h = hex_color.lstrip('#')
        rgb_c = tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
        colored[r_ch == cls_id] = rgb_c
    
    # RGB와 Semantic 오버레이
    overlay = (rgb * 0.5 + colored * 0.5).astype(np.uint8)
    ax.imshow(overlay)
    ax.set_title(f.stem, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(OUT_DIR / 'semantic_gt_samples.png', dpi=100, bbox_inches='tight')
plt.show()
print('GT 샘플 시각화 완료')

## Cell 2: 학습용 클래스 정의 (클래스 병합)

28개 CARLA 클래스를 모두 학습하면 데이터가 너무 적음.
**핵심 6개 클래스만** 학습 (자율주행에서 중요한 클래스 위주).

In [ ]:
# ── 학습 클래스 정의 ──────────────────────────────────────────────────────
# CARLA class ID → 학습 class ID 매핑
# 자율주행에서 중요한 6개 클래스 (+ background)
TRAIN_CLASSES = {
    'background': 0,  # 나머지 모두
    'road':       1,  # CARLA 7
    'sidewalk':   2,  # CARLA 8
    'vehicle':    3,  # CARLA 10, 28
    'pedestrian': 4,  # CARLA 4
    'vegetation': 5,  # CARLA 9
    'sky':        6,  # CARLA 13
}
NUM_CLASSES = len(TRAIN_CLASSES)  # 7

# CARLA → Train 클래스 매핑 테이블
CARLA_TO_TRAIN = np.zeros(256, dtype=np.uint8)  # default = 0 (background)
CARLA_TO_TRAIN[7]  = 1   # Road
CARLA_TO_TRAIN[8]  = 2   # SideWalk
CARLA_TO_TRAIN[10] = 3   # Vehicles
CARLA_TO_TRAIN[28] = 3   # Car (CARLA 0.9.15+)
CARLA_TO_TRAIN[4]  = 4   # Pedestrian
CARLA_TO_TRAIN[9]  = 5   # Vegetation
CARLA_TO_TRAIN[13] = 6   # Sky

# 클래스 컬러 (시각화용)
CLASS_COLORS = {
    0: (50,  50,  50),   # background (dark gray)
    1: (128, 64,  128),  # road (purple)
    2: (244, 35,  232),  # sidewalk (pink)
    3: (0,   0,   142),  # vehicle (dark blue)
    4: (220, 20,  60),   # pedestrian (red)
    5: (107, 142, 35),   # vegetation (green)
    6: (70,  130, 180),  # sky (light blue)
}
CLASS_NAMES = list(TRAIN_CLASSES.keys())

def carla_mask_to_train(sem_bgr):
    """CARLA semantic PNG(BGR) → 학습용 mask (H,W) uint8"""
    r_ch = sem_bgr[:, :, 2]  # R채널 = CARLA class ID
    return CARLA_TO_TRAIN[r_ch]

def mask_to_color(mask):
    """학습용 mask (H,W) → RGB 컬러 이미지"""
    colored = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls_id, color in CLASS_COLORS.items():
        colored[mask == cls_id] = color
    return colored

# ── 매핑 확인 시각화 ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
f = random.choice(sem_files)

rgb     = cv2.cvtColor(cv2.imread(str(IMG_DIR / f'{f.stem}.jpg')), cv2.COLOR_BGR2RGB)
sem_bgr = cv2.imread(str(f))
train_mask = carla_mask_to_train(sem_bgr)

axes[0].imshow(rgb)
axes[0].set_title('원본 RGB')
axes[0].axis('off')

axes[1].imshow(sem_bgr[:,:,2], cmap='tab20', vmin=0, vmax=28)
axes[1].set_title('CARLA 원본 Semantic (28 클래스)')
axes[1].axis('off')

axes[2].imshow(mask_to_color(train_mask))
# 범례 추가
patches_list = [mpatches.Patch(color=np.array(c)/255, label=CLASS_NAMES[i])
                for i, c in CLASS_COLORS.items()]
axes[2].legend(handles=patches_list, loc='lower right', fontsize=7)
axes[2].set_title('학습용 Semantic (7 클래스)')
axes[2].axis('off')

plt.tight_layout()
plt.savefig(OUT_DIR / 'class_mapping.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'학습 클래스: {NUM_CLASSES}개 — {CLASS_NAMES}')

## Cell 3: CARLA Semantic Dataset + DataLoader

In [ ]:
from torchvision import transforms as T

# ── Dataset 클래스 ────────────────────────────────────────────────────────
class CarlaSegDataset(Dataset):
    """
    CARLA RGB + Semantic Mask 데이터셋.
    RGB image → (C,H,W) float32 텐서
    Mask      → (H,W) long 텐서 (0~NUM_CLASSES-1)
    """
    IMG_MEAN = [0.485, 0.456, 0.406]  # ImageNet 정규화
    IMG_STD  = [0.229, 0.224, 0.225]
    
    def __init__(self, img_dir, sem_dir, frame_ids, img_size=(512, 512), augment=False):
        self.img_dir  = Path(img_dir)
        self.sem_dir  = Path(sem_dir)
        self.frame_ids = frame_ids
        self.img_size  = img_size  # (H, W)
        self.augment   = augment
        
        self.normalize = T.Normalize(mean=self.IMG_MEAN, std=self.IMG_STD)
    
    def __len__(self):
        return len(self.frame_ids)
    
    def __getitem__(self, idx):
        fid = self.frame_ids[idx]
        
        # RGB 로드
        img_bgr = cv2.imread(str(self.img_dir / f'{fid}.jpg'))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        
        # Semantic 마스크 로드
        sem_bgr = cv2.imread(str(self.sem_dir / f'{fid}.png'))
        mask    = carla_mask_to_train(sem_bgr)  # (H,W) uint8
        
        # 리사이즈
        H, W = self.img_size
        img_rgb = cv2.resize(img_rgb, (W, H), interpolation=cv2.INTER_LINEAR)
        mask    = cv2.resize(mask,    (W, H), interpolation=cv2.INTER_NEAREST)
        
        # 데이터 증강 (랜덤 좌우 뒤집기)
        if self.augment and random.random() > 0.5:
            img_rgb = np.fliplr(img_rgb).copy()
            mask    = np.fliplr(mask).copy()
        
        # 색상 지터 (밝기/대비 변화 — 조명 변화 모의)
        if self.augment and random.random() > 0.5:
            factor = np.random.uniform(0.7, 1.3)
            img_rgb = np.clip(img_rgb * factor, 0, 255).astype(np.uint8)
        
        # numpy → tensor
        img_tensor  = torch.from_numpy(img_rgb).permute(2, 0, 1).float() / 255.0
        img_tensor  = self.normalize(img_tensor)  # ImageNet 정규화
        mask_tensor = torch.from_numpy(mask).long()
        
        return img_tensor, mask_tensor

# ── Train/Val 분할 ────────────────────────────────────────────────────────
all_frame_ids = [f.stem for f in sorted(SEM_DIR.glob('*.png'))
                 if (IMG_DIR / f.stem).with_suffix('.jpg').exists()]
random.seed(42)
random.shuffle(all_frame_ids)

n_train = int(len(all_frame_ids) * 0.8)
train_ids = all_frame_ids[:n_train]
val_ids   = all_frame_ids[n_train:]

print(f'전체 프레임: {len(all_frame_ids)} / Train: {len(train_ids)} / Val: {len(val_ids)}')

IMG_SIZE = (512, 512)  # SegFormer 권장 입력 크기
BATCH    = 4           # 512×512 × batch4 → ~6GB VRAM (RTX 4080 Super 16GB 여유)

train_ds = CarlaSegDataset(IMG_DIR, SEM_DIR, train_ids, IMG_SIZE, augment=True)
val_ds   = CarlaSegDataset(IMG_DIR, SEM_DIR, val_ids,   IMG_SIZE, augment=False)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)

# 샘플 확인
imgs, masks = next(iter(train_dl))
print(f'Image batch: {imgs.shape} / Mask batch: {masks.shape}')
print(f'Mask unique classes: {masks.unique().tolist()}')

## Cell 4: SegFormer 모델 로드 + 헤드 교체

### SegFormer 구조 (논문 핵심)
```
Input → MiT Encoder (계층적 Transformer) → MLP Decoder → Segmentation Map
         └ 4단계 피처 추출 (1/4, 1/8, 1/16, 1/32 해상도)
                                    └ 모든 스케일 합쳐서 예측
```
- **MiT (Mix Transformer)**: 이미지를 패치로 분할 → 계층별 어텐션
- **MLP Decoder**: CNN 디코더 대신 단순 MLP → 경량화
- **b0 variant**: 3.8M params, ADE20K mIoU 37.4

In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerConfig

# ── SegFormer-b0 로드 (ADE20K pretrained → 150 클래스) ────────────────────
print('SegFormer-b0 로드 중...')
model = SegformerForSemanticSegmentation.from_pretrained(
    'nvidia/segformer-b0-finetuned-ade-512-512',
    num_labels=NUM_CLASSES,          # 7 클래스로 헤드 교체
    id2label={i: n for i, n in enumerate(CLASS_NAMES)},
    label2id={n: i for i, n in enumerate(CLASS_NAMES)},
    ignore_mismatched_sizes=True,    # 헤드 크기 불일치 무시 (새로 초기화)
)
model = model.to(DEVICE)

# 파라미터 수
total_params   = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n전체 파라미터: {total_params/1e6:.1f}M')
print(f'학습 파라미터: {trainable_params/1e6:.1f}M')
print()

# 구조 요약
print('=== SegFormer-b0 구조 ===')
print('Encoder: Mix Transformer (MiT-b0)')
print('  - 4단계 계층적 Transformer 블록')
print('  - Overlap Patch Merging (위치 인코딩 대체)')
print('  - Efficient Self-Attention (해상도별 축소)')
print('Decoder: All-MLP Head')
print('  - 4개 스케일 피처 → 1/4 해상도 예측')
print(f'  - 출력 클래스: {NUM_CLASSES} ({CLASS_NAMES})')
print('Input: 512×512 (ADE20K 표준)')
print('Pretrain: ADE20K (150 클래스) → CARLA 7 클래스로 헤드 교체')

## Cell 5: 파인튜닝 학습 루프

### 전략
- **백본(MiT)**: 낮은 학습률 (1e-4) — pretrained feature 보존
- **디코더 헤드**: 높은 학습률 (6e-4) — CARLA 도메인 빠른 적응
- **Loss**: Cross-Entropy (클래스 불균형 → class weight 적용)

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── 클래스 가중치 계산 (Inverse Frequency) ────────────────────────────────
# 픽셀이 적은 클래스(차량, 보행자)에 높은 가중치 부여
print('클래스 가중치 계산 중 (전체 train set)...')
cls_counts = np.zeros(NUM_CLASSES)
for fid in train_ids:  # 전체 train set 사용
    sem_bgr = cv2.imread(str(SEM_DIR / f'{fid}.png'))
    mask = carla_mask_to_train(sem_bgr)
    for c in range(NUM_CLASSES):
        cls_counts[c] += (mask == c).sum()

# Median Frequency Balancing (inverse frequency 대비 안정적)
# freq(c) = 해당 클래스 픽셀 수 / 전체 픽셀 수
# weight(c) = median(freq) / freq(c)
# → 중앙값 클래스 weight=1.0, 희귀 클래스는 높게, 지배적 클래스는 낮게
cls_counts = np.maximum(cls_counts, 1)
freq = cls_counts / cls_counts.sum()
median_freq = np.median(freq)
weights = median_freq / freq
weights = np.clip(weights, 0.1, 5.0)  # 극단값 방지
class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

print('클래스별 가중치:')
for i, (name, w) in enumerate(zip(CLASS_NAMES, weights)):
    print(f'  {name:>12}: {w:.4f}')

# ── 옵티마이저: 백본/헤드 차등 학습률 ────────────────────────────────────
EPOCHS  = 30
LR_BACKBONE = 1e-4
LR_HEAD     = 6e-4

optimizer = AdamW([
    {'params': model.segformer.parameters(), 'lr': LR_BACKBONE},  # MiT 백본
    {'params': model.decode_head.parameters(), 'lr': LR_HEAD},    # MLP 헤드
], weight_decay=1e-2)

scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# ── mIoU 계산 함수 ────────────────────────────────────────────────────────
def compute_miou(preds, targets, num_classes):
    """
    preds  : (N, H, W) long tensor — 예측 클래스
    targets: (N, H, W) long tensor — 정답 클래스
    """
    ious = []
    preds_np   = preds.cpu().numpy().flatten()
    targets_np = targets.cpu().numpy().flatten()
    for c in range(num_classes):
        pred_c = (preds_np == c)
        gt_c   = (targets_np == c)
        inter  = (pred_c & gt_c).sum()
        union  = (pred_c | gt_c).sum()
        if union == 0:
            continue  # 해당 클래스 없으면 스킵
        ious.append(inter / union)
    return float(np.mean(ious)) if ious else 0.0

# ── 학습 루프 ─────────────────────────────────────────────────────────────
print(f'\n학습 시작: {EPOCHS} epoch, batch={BATCH}, img={IMG_SIZE}')
print(f'  백본 lr={LR_BACKBONE} / 헤드 lr={LR_HEAD}')
print()

history = {'train_loss': [], 'val_loss': [], 'val_miou': []}
best_miou = 0.0
best_epoch = 0

for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    train_losses = []
    for imgs, masks in train_dl:
        imgs  = imgs.to(DEVICE)
        masks = masks.to(DEVICE)
        
        outputs = model(pixel_values=imgs, labels=masks)
        # SegFormer loss = CrossEntropy (내부 계산), 또는 직접 계산
        # logits: (N, num_classes, H/4, W/4) → upsample
        logits = outputs.logits  # (N, C, H', W')
        logits_up = F.interpolate(logits, size=masks.shape[-2:],
                                  mode='bilinear', align_corners=False)
        loss = F.cross_entropy(logits_up, masks, weight=class_weights)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    
    scheduler.step()
    
    # ── Validation ──
    model.eval()
    val_losses = []
    all_preds, all_targets = [], []
    with torch.no_grad():
        for imgs, masks in val_dl:
            imgs  = imgs.to(DEVICE)
            masks = masks.to(DEVICE)
            logits = model(pixel_values=imgs).logits
            logits_up = F.interpolate(logits, size=masks.shape[-2:],
                                      mode='bilinear', align_corners=False)
            loss = F.cross_entropy(logits_up, masks, weight=class_weights)
            val_losses.append(loss.item())
            preds = logits_up.argmax(dim=1)
            all_preds.append(preds.cpu())
            all_targets.append(masks.cpu())
    
    all_preds   = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    val_miou    = compute_miou(all_preds, all_targets, NUM_CLASSES)
    
    t_loss = np.mean(train_losses)
    v_loss = np.mean(val_losses)
    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['val_miou'].append(val_miou)
    
    if val_miou > best_miou:
        best_miou  = val_miou
        best_epoch = epoch + 1
        torch.save(model.state_dict(), OUT_DIR / 'segformer_carla_best.pth')
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | '
              f'Train Loss: {t_loss:.4f} | '
              f'Val Loss: {v_loss:.4f} | '
              f'Val mIoU: {val_miou:.4f}'
              f'{" ← BEST" if val_miou == best_miou else ""}')

print(f'\n학습 완료! Best mIoU: {best_miou:.4f} (Epoch {best_epoch})')

## Cell 6: 클래스별 IoU 상세 평가 + SAM2 비교

In [ ]:
# ── Best 모델 로드 ────────────────────────────────────────────────────────
model.load_state_dict(torch.load(OUT_DIR / 'segformer_carla_best.pth', map_location=DEVICE))
model.eval()

# ── 클래스별 IoU 계산 ─────────────────────────────────────────────────────
confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

with torch.no_grad():
    for imgs, masks in val_dl:
        imgs   = imgs.to(DEVICE)
        logits = model(pixel_values=imgs).logits
        logits_up = F.interpolate(logits, size=masks.shape[-2:],
                                  mode='bilinear', align_corners=False)
        preds = logits_up.argmax(dim=1).cpu().numpy().flatten()
        gts   = masks.numpy().flatten()
        for p, g in zip(preds, gts):
            if 0 <= g < NUM_CLASSES:
                confusion[g, p] += 1

# 클래스별 IoU
per_class_iou = {}
for c in range(NUM_CLASSES):
    tp    = confusion[c, c]
    fp    = confusion[:, c].sum() - tp
    fn    = confusion[c, :].sum() - tp
    union = tp + fp + fn
    iou   = tp / union if union > 0 else 0.0
    per_class_iou[CLASS_NAMES[c]] = float(iou)

miou_segformer = np.mean(list(per_class_iou.values()))

# SAM2 Phase 4 결과 (비교용)
SAM2_MIOU     = 0.1070
SAM2_CAR_IOU  = 0.0064
SAM2_PERS_IOU = 0.1302

print('=' * 55)
print('클래스별 IoU — SegFormer (CARLA 파인튜닝)')
print('=' * 55)
for name, iou in per_class_iou.items():
    bar = '█' * int(iou * 20)
    print(f'  {name:>12}: {iou:.4f}  {bar}')
print('-' * 55)
print(f'  {"mIoU":>12}: {miou_segformer:.4f}')
print('=' * 55)
print()
print('=== SAM2 vs SegFormer 비교 ===')
print(f'  SAM2 (Phase 4) mIoU  : {SAM2_MIOU:.4f}')
print(f'  SegFormer (CARLA ft) : {miou_segformer:.4f}  ({miou_segformer - SAM2_MIOU:+.4f})')
print()
print('  [car]')
print(f'    SAM2     : {SAM2_CAR_IOU:.4f}')
print(f'    SegFormer: {per_class_iou.get("vehicle", 0):.4f}')
print('  [pedestrian]')
print(f'    SAM2     : {SAM2_PERS_IOU:.4f}')
print(f'    SegFormer: {per_class_iou.get("pedestrian", 0):.4f}')

## Cell 7: 예측 시각화 + 학습 곡선

In [ ]:
# ── 학습 곡선 ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs_range = range(1, EPOCHS + 1)
axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss',   color='coral')
axes[0].set_title('Loss 학습 곡선')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['val_miou'], color='mediumseagreen', linewidth=2)
axes[1].axhline(y=SAM2_MIOU, color='red', linestyle='--', label=f'SAM2 기준 ({SAM2_MIOU:.3f})')
axes[1].axhline(y=miou_segformer, color='green', linestyle='--', 
                label=f'Best SegFormer ({miou_segformer:.3f})')
axes[1].set_title('Val mIoU 학습 곡선')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('mIoU')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'segformer_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 예측 결과 시각화 ──────────────────────────────────────────────────────
sample_val_ids = random.sample(val_ids, 4)
fig, axes = plt.subplots(4, 3, figsize=(15, 16))
fig.suptitle('SegFormer 예측 결과: RGB | GT | Prediction', fontsize=13)

IMG_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
IMG_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

with torch.no_grad():
    for row, fid in enumerate(sample_val_ids):
        img_path = IMG_DIR / f'{fid}.jpg'
        sem_path = SEM_DIR / f'{fid}.png'
        
        img_bgr = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rgb_r = cv2.resize(img_rgb, (512, 512))
        
        sem_bgr = cv2.imread(str(sem_path))
        gt_mask = carla_mask_to_train(sem_bgr)
        gt_mask_r = cv2.resize(gt_mask, (512, 512), interpolation=cv2.INTER_NEAREST)
        
        # 추론
        img_t = torch.from_numpy(img_rgb_r).permute(2,0,1).float() / 255.0
        img_t = (img_t - IMG_MEAN) / IMG_STD
        img_t = img_t.unsqueeze(0).to(DEVICE)
        
        logits = model(pixel_values=img_t).logits
        logits_up = F.interpolate(logits, size=(512,512), mode='bilinear', align_corners=False)
        pred_mask = logits_up.argmax(dim=1)[0].cpu().numpy()
        
        axes[row, 0].imshow(img_rgb_r)
        axes[row, 0].set_title(f'RGB ({fid})', fontsize=9)
        axes[row, 0].axis('off')
        
        axes[row, 1].imshow(mask_to_color(gt_mask_r))
        axes[row, 1].set_title('GT Semantic', fontsize=9)
        axes[row, 1].axis('off')
        
        axes[row, 2].imshow(mask_to_color(pred_mask))
        axes[row, 2].set_title('SegFormer 예측', fontsize=9)
        axes[row, 2].axis('off')

# 범례
patches_list = [mpatches.Patch(color=np.array(c)/255, label=CLASS_NAMES[i])
                for i, c in CLASS_COLORS.items()]
fig.legend(handles=patches_list, loc='lower center', ncol=7, fontsize=9, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig(OUT_DIR / 'segformer_predictions.png', dpi=100, bbox_inches='tight')
plt.show()
print('시각화 완료')

## Cell 8: 최종 요약 대시보드

In [ ]:
# ── Option A+B 통합 결과 대시보드 ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('스킬업 라운드 2: Option A + B 결과', fontsize=14)

# 1. Detection mAP 개선 (Option A)
opt_a = {'베이스라인\n(COCO)': 0.4269, '파인튜닝\n(CARLA)': 0.6844}
bars = axes[0].bar(opt_a.keys(), opt_a.values(),
                   color=['steelblue', 'coral'], alpha=0.85)
axes[0].set_title('[Option A] Detection mAP@50')
axes[0].set_ylim(0, 1.0)
axes[0].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, opt_a.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2., val + 0.02,
                 f'{val:.3f}', ha='center', fontweight='bold')

# 2. Segmentation mIoU 개선 (Option B)
opt_b = {'SAM2\n(Phase 4)': SAM2_MIOU, 'SegFormer\n(CARLA ft)': miou_segformer}
bars2 = axes[1].bar(opt_b.keys(), opt_b.values(),
                    color=['steelblue', 'mediumseagreen'], alpha=0.85)
axes[1].set_title('[Option B] Segmentation mIoU')
axes[1].set_ylim(0, 1.0)
axes[1].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars2, opt_b.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2., val + 0.02,
                 f'{val:.3f}', ha='center', fontweight='bold')

# 3. 클래스별 IoU (SegFormer)
cls_names_plot = list(per_class_iou.keys())
cls_iou_plot   = list(per_class_iou.values())
colors_plot    = [np.array(CLASS_COLORS[i])/255 for i in range(NUM_CLASSES)]
bars3 = axes[2].barh(cls_names_plot, cls_iou_plot, color=colors_plot, edgecolor='black', linewidth=0.5)
axes[2].set_title('[Option B] 클래스별 IoU')
axes[2].set_xlim(0, 1.0)
axes[2].axvline(x=miou_segformer, color='black', linestyle='--', alpha=0.7, label=f'mIoU={miou_segformer:.3f}')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars3, cls_iou_plot):
    if val > 0.02:
        axes[2].text(val + 0.01, bar.get_y() + bar.get_height()/2.,
                     f'{val:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / 'option_ab_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 텍스트 최종 요약 ──────────────────────────────────────────────────────
print()
print('=' * 60)
print('스킬업 라운드 2 — Option A + B 최종 요약')
print('=' * 60)
print('[Option A] YOLOv8 CARLA 도메인 적응')
print(f'  mAP@50  : 0.4269 → 0.6844  (+0.2575)')
print(f'  Precision: 0.4931 → 0.8500  (+0.3569)')
print(f'  MOTA    : -0.6863 → 0.2941  (+0.9804)  Detection이 병목 확인')
print()
print('[Option B] SegFormer CARLA Semantic Segmentation')
print(f'  SAM2 mIoU  : {SAM2_MIOU:.4f}  (태스크 미스매치)')
print(f'  SegFormer  : {miou_segformer:.4f}  ({miou_segformer-SAM2_MIOU:+.4f})')
print(f'  vehicle IoU: {per_class_iou.get("vehicle", 0):.4f}')
print(f'  person  IoU: {per_class_iou.get("pedestrian", 0):.4f}')
print()
print('→ 다음: Option C — Depth 스케일 정렬 + RMSE 개선')
print('=' * 60)